# Door 2: mixture densities to a binned fraction measurement

This notebook accompanies the docs page [`door2-mixture-densities`](../../docs/examples/door2-mixture-densities.md). A one-dimensional spectrum is a mixture of a signal peak and a truncated-exponential background, both exact normalized densities on $[0, 1]$. The notebook builds the linear component score explicitly, fits from both a Monte Carlo sample and a bounded `IntegrationSource`, and measures the retained Fisher information about the signal fraction specifically, at a larger sample and finer quadrature than the docs page's fast snippets use.

## Data

`examples.synthetic_problems.signal_background_shape` returns the exact component densities and precomputed Monte Carlo splits. One background shape keeps this a clean two-parameter problem.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import scorequant as sq
from examples._env import example_scale
from examples.synthetic_problems import signal_background_shape

sizes = (example_scale(8_000, 800), example_scale(2_000, 400), example_scale(15_000, 1_500))
problem = signal_background_shape(background_rates=(2.5,), n_bins=6, sizes=sizes)
problem.component_names, problem.interest, problem.nuisance

## Component pdfs to a linear component score

Wrapping the exact densities as `LinearComponents` makes the component-to-score step an explicit API call, and `LinearComponentScore` reproduces the generator's own precomputed scores exactly.

In [ ]:
def signal_component(x: np.ndarray) -> np.ndarray:
    return problem.signal_density(np.asarray(x)[:, 0])


def background_component(x: np.ndarray) -> np.ndarray:
    return problem.background_densities[0](np.asarray(x)[:, 0])


model = sq.LinearComponents(
    components={"signal": signal_component, "background": background_component},
    coefficients={
        "signal": float(problem.coefficients[0]),
        "background": float(problem.coefficients[1]),
    },
    variables=["x"],
)
provider = sq.LinearComponentScore(model)
np.allclose(np.asarray(provider.score(problem.train.observations)), problem.train.scores)

In [ ]:
grid = np.linspace(0.0, 1.0, 400)[:, None]
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(grid[:, 0], problem.signal_density(grid[:, 0]), label="signal shape")
ax.plot(grid[:, 0], problem.background_densities[0](grid[:, 0]), label="background shape")
ax.set(xlabel="x", ylabel="density", title="Component pdfs")
ax.legend();

## Two routes to the same fit

A Monte Carlo `ObservationSample` and a bounded `IntegrationSource` both reach `fit_quantizer` through the same `provider`. The `IntegrationSource` path carries two score columns end to end — one per component — with no Monte Carlo sample at all.

In [ ]:
train, test = problem.train, problem.test

quantizer_mc = sq.fit_quantizer(
    sq.ObservationSample(train.observations, train.weights),
    provider=provider,
    n_bins=problem.n_bins,
    criterion=sq.DOptimality(),
    config=sq.DExchangeConfig(seed=50, n_init=8),
)

source = sq.IntegrationSource(
    problem.bounds, density=problem.intensity, quadrature=sq.GaussLegendreConfig(order=96)
)
quantizer_int = sq.fit_quantizer(
    source,
    provider=provider,
    n_bins=problem.n_bins,
    criterion=sq.DOptimality(),
    config=sq.DExchangeConfig(seed=50, n_init=8),
)

mc_retention = quantizer_mc.evaluate_scores(test.scores, test.weights).geometric_mean_retention
int_retention = quantizer_int.evaluate_scores(test.scores, test.weights).geometric_mean_retention
quantizer_int.source_kind, float(mc_retention), float(int_retention)

## The downstream payoff: information about the fraction specifically

The overall D-efficiency compresses both coefficients jointly. `profiled_information_report` Schur-completes the nuisance background direction out of both the unbinned and the binned Fisher matrix, isolating what the six bins retained about the signal fraction alone.

In [ ]:
labels = np.asarray(quantizer_int.predict_scores(test.scores))
profiled = sq.profiled_information_report(
    test.scores, labels, interest=problem.interest, weights=test.weights, n_bins=problem.n_bins
)
{
    "unbinned Fisher (signal fraction)": float(profiled.schur_unbinned[0, 0]),
    "binned Fisher (signal fraction)": float(profiled.schur_binned[0, 0]),
    "retained fraction": float(profiled.geometric_mean_retention),
}

## Interpretation

The Monte Carlo and `IntegrationSource` fits agree closely, and both retain over 99% of the Fisher information about the signal fraction after binning. This page treats both coefficients jointly with plain `DOptimality`; when the background shape should instead be optimized around as a genuine nuisance rather than just diagnosed afterward, that calls for `ProfiledDOptimality` — see a later profiled-$D_s$ page for that comparison on this same generator.